In [ ]:
from __future__ import annotations
import os, gc, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models

In [ ]:
class CFG:
    COMP_DIR = Path("/kaggle/input/competitions/birdclef-2026")
    TRAIN_AUDIO_DIR = COMP_DIR / "train_audio"
    TRAIN_CSV = COMP_DIR / "train.csv"
    TEST_DIR = COMP_DIR / "test_soundscapes"
    SAMPLE_SUB = COMP_DIR / "sample_submission.csv"
    OUTPUT_DIR = Path("/kaggle/working")

    SR = 16000              # ↓ reduced
    DURATION = 2            # ↓ faster
    N_SAMPLES = SR * DURATION

    N_MELS = 64             # ↓ faster
    EPOCHS = 1
    BATCH_SIZE = 16
    NUM_WORKERS = 2

    DEBUG = True
    N_DEBUG = 100
    RUN_INFERENCE = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_df = pd.read_csv(CFG.TRAIN_CSV)
sample_sub = pd.read_csv(CFG.SAMPLE_SUB)

SPECIES = [c for c in sample_sub.columns if c not in ("row_id","filename","end_time")]
NUM_CLASSES = len(SPECIES)
SPECIES2IDX = {s:i for i,s in enumerate(SPECIES)}

if CFG.DEBUG:
    train_df = train_df.sample(n=min(CFG.N_DEBUG,len(train_df)), random_state=42)

train_df["filepath"] = train_df["filename"].apply(lambda x: str(CFG.TRAIN_AUDIO_DIR / x))
train_df = train_df[train_df["filepath"].map(lambda p: Path(p).exists())]

def encode(row):
    y = np.zeros(NUM_CLASSES, dtype=np.float32)
    if row["primary_label"] in SPECIES2IDX:
        y[SPECIES2IDX[row["primary_label"]]] = 1
    return y

train_df["label"] = train_df.apply(encode, axis=1)

In [ ]:
def load_audio(path):
    try:
        y,_ = librosa.load(path, sr=CFG.SR)
    except:
        y = np.zeros(CFG.N_SAMPLES)

    if len(y) < CFG.N_SAMPLES:
        y = np.pad(y,(0,CFG.N_SAMPLES-len(y)))
    return y[:CFG.N_SAMPLES]

def mel_spec(y):
    m = librosa.feature.melspectrogram(y=y, sr=CFG.SR, n_mels=CFG.N_MELS)
    m = librosa.power_to_db(m)
    return (m+80)/80

In [ ]:
class DS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        y = load_audio(row["filepath"])
        m = mel_spec(y)
        m = torch.tensor(m).unsqueeze(0)
        label = torch.tensor(row["label"])
        return m, label

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, NUM_CLASSES)

    def forward(self,x):
        x = x.repeat(1,3,1,1)
        return self.backbone(x)

In [ ]:
def train():
    ds = DS(train_df)
    dl = DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS)

    model = Model().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()

    model.train()
    for ep in range(CFG.EPOCHS):
        t0=time.time()
        for x,y in dl:
            x,y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = loss_fn(out,y)

            opt.zero_grad()
            loss.backward()
            opt.step()
        print(f"Epoch {ep+1} done in {time.time()-t0:.1f}s")

    torch.save(model.state_dict(), CFG.OUTPUT_DIR/"model.pt")
    return model

In [ ]:
@torch.no_grad()
def predict(model, path):
    y = load_audio(path)
    m = mel_spec(y)
    m = torch.tensor(m).unsqueeze(0).unsqueeze(0).to(DEVICE)
    out = torch.sigmoid(model(m))[0].cpu().numpy()
    return out


In [ ]:
def run_inference(model):
    sub = pd.read_csv(CFG.SAMPLE_SUB)

    # 🔒 guarantee exact structure
    expected_cols = sub.columns.tolist()

    test_files = list(CFG.TEST_DIR.glob("*.ogg"))

    # ===== CASE 1: No test files (local / debug) =====
    if len(test_files) == 0:
        print("No test files → using sample_submission format")

        for col in expected_cols:
            if col not in ["row_id", "filename", "end_time"]:
                sub[col] = 0.0

        sub = sub[expected_cols]  # enforce order
        sub.fillna(0.0, inplace=True)

        sub.to_csv(CFG.OUTPUT_DIR / "submission.csv", index=False)
        return

    # ===== CASE 2: Real inference (Kaggle hidden test) =====
    rows = []

    for f in test_files:
        probs = predict(model, f)

        for t in range(5, 65, 5):
            row = {
                "row_id": f"{f.name}_{t}",
                "filename": f.name,
                "end_time": t
            }

            for i, sp in enumerate(SPECIES):
                row[sp] = float(probs[i])

            rows.append(row)

    sub_pred = pd.DataFrame(rows)

    # 🔒 HARD ALIGNMENT WITH SAMPLE SUBMISSION
    sub_final = sub.copy()

    for col in expected_cols:
        if col not in sub_pred.columns:
            sub_pred[col] = 0.0

    sub_pred = sub_pred[expected_cols]

    # Match row count EXACTLY
    if len(sub_pred) != len(sub_final):
        print("Row mismatch → forcing sample format")
        sub_final.loc[:, expected_cols[3:]] = 0.0
        sub_final.to_csv(CFG.OUTPUT_DIR / "submission.csv", index=False)
        return

    sub_pred.fillna(0.0, inplace=True)
    sub_pred.to_csv(CFG.OUTPUT_DIR / "submission.csv", index=False)


In [ ]:
def main():
    model = train()
    if CFG.RUN_INFERENCE:
        run_inference(model)

if __name__=="__main__":
    main()